# Notebook 02: Optimization and Common Evaluation (clean v2)

This notebook uses the finalized outputs from Notebook 01 v7 and runs the two-stage stochastic CVaR supply-chain optimization, common evaluation, robustness checks, and sensitivity analyses. It creates node parameters and transport arcs automatically if those files are not present.

In [ ]:
# Reusable implementation imports
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hurricane_resilience.data import load_event_node_data, read_csv, haversine_km
from hurricane_resilience.evaluation import weighted_cvar, extract_nondominated
from hurricane_resilience.paths import FIGURES_DIR, TABLES_DIR
from hurricane_resilience.scenarios import validate_scenario_weights, kmeans_reduce


In [ ]:
# Cell 00: Global setup, paths, reproducibility, and file utilities

import os
import re
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
NOTEBOOK_VERSION = "02_clean_v2_auto_node_params"

# Run this notebook from the same project folder as 01_data_ml_pipeline_clean_v7.ipynb outputs.
BASE_DIR = PROJECT_ROOT

# If your notebook is opened from another folder, manually set BASE_DIR here, for example:
# PROJECT_ROOT is resolved above from this notebook location

OUTPUT_DIR = BASE_DIR / "outputs_02_optimization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DPI = 300

# If False, the notebook loads existing output files whenever available.
# If True, it recomputes optimization results even when output files already exist.
# For a complete from-scratch rerun, keep only the 01 input files and set FORCE_RECOMPUTE = True.
FORCE_RECOMPUTE = False

print("Notebook version:", NOTEBOOK_VERSION)
print("Working directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("FORCE_RECOMPUTE:", FORCE_RECOMPUTE)


def find_file(filename, patterns=None, required=True):
    """Find a file in BASE_DIR or OUTPUT_DIR, with optional fallback glob patterns."""
    search_dirs = [BASE_DIR, OUTPUT_DIR]
    for folder in search_dirs:
        exact = folder / filename
        if exact.exists():
            return exact

    candidates = []
    if patterns:
        for folder in search_dirs:
            for pattern in patterns:
                candidates.extend(folder.glob(pattern))
    candidates = sorted(set(candidates))

    if candidates:
        print(f"Using fallback for {filename}: {candidates[0]}")
        return candidates[0]

    if required:
        available = sorted([p.name for p in BASE_DIR.glob("*.csv")])
        preview = "\n".join(available[:120])
        raise FileNotFoundError(
            f"Cannot find required file: {filename}\n"
            f"Searched folders:\n- {BASE_DIR}\n- {OUTPUT_DIR}\n"
            f"Fallback patterns: {patterns}\n"
            f"Available CSV files in BASE_DIR include:\n{preview}"
        )
    return None


def save_csv(df, filename, also_root=True):
    """Save CSV to OUTPUT_DIR and, optionally, to BASE_DIR using the legacy manuscript filename."""
    out_path = OUTPUT_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    if also_root:
        root_path = BASE_DIR / filename
        df.to_csv(root_path, index=False, encoding="utf-8-sig")
    print("Saved:", out_path)
    return out_path


def maybe_load_csv(filename):
    """Load existing output if FORCE_RECOMPUTE is False and the file exists in BASE_DIR or OUTPUT_DIR."""
    if FORCE_RECOMPUTE:
        return None
    path = find_file(filename, required=False)
    if path is not None:
        print("Loaded existing:", path)
        return pd.read_csv(path)
    return None


def display_if_available(obj, max_rows=20):
    """Display a dataframe in notebooks; fallback to print in plain Python."""
    if isinstance(obj, pd.DataFrame):
        try:
            display(obj.head(max_rows))
        except NameError:
            print(obj.head(max_rows).to_string())
    else:
        try:
            display(obj)
        except NameError:
            print(obj)

In [ ]:
# Cell 01: Load and validate Notebook 01 outputs

RISK_MATRIX_FILE = "final_optimization_scenario_node_risk_matrix_without_season.csv"
FULL_PREDICTION_FILE = "final_risk_predictions_oof_without_season.csv"
HYBRID_SCENARIO_FILE = "final_hybrid_reduced_hurricane_scenarios_without_season.csv"
STORM_NODE_FILE = "storm_node_dataset_14nodes_200km_50kt.csv"

risk_matrix_path = find_file(RISK_MATRIX_FILE, patterns=["*optimization*scenario*node*risk*without_season*.csv"])
full_prediction_path = find_file(FULL_PREDICTION_FILE, patterns=["*risk_predictions*oof*without_season*.csv"])
hybrid_scenario_path = find_file(HYBRID_SCENARIO_FILE, patterns=["*hybrid*reduced*hurricane*without_season*.csv"])
storm_node_path = find_file(STORM_NODE_FILE, patterns=["*storm*node*14nodes*200km*50kt*.csv", "*stom*node*14nodes*200km*50kt*.csv"], required=False)

risk_matrix = pd.read_csv(risk_matrix_path)
full_predictions = pd.read_csv(full_prediction_path)
hybrid_scenarios = pd.read_csv(hybrid_scenario_path)
storm_node_df = pd.read_csv(storm_node_path) if storm_node_path is not None else full_predictions.copy()

required_risk_columns = {
    "scenario_id", "SEASON", "NAME", "node_id", "node_name", "node_type",
    "p_is", "p_is_sensitivity", "disrupted", "scenario_weight"
}
missing = sorted(required_risk_columns - set(risk_matrix.columns))
if missing:
    raise ValueError(f"Risk matrix is missing required columns: {missing}")

if risk_matrix["scenario_id"].nunique() != hybrid_scenarios["scenario_id"].nunique():
    raise ValueError(
        "Risk matrix scenario count does not match hybrid scenario file. "
        f"risk_matrix={risk_matrix['scenario_id'].nunique()}, "
        f"hybrid={hybrid_scenarios['scenario_id'].nunique()}"
    )

nodes_per_scenario = risk_matrix.groupby("scenario_id")["node_id"].nunique()
if not (nodes_per_scenario == 14).all():
    bad = nodes_per_scenario[nodes_per_scenario != 14]
    raise ValueError(f"Every scenario must contain exactly 14 nodes. Bad scenarios:\n{bad}")

scenario_weight_sum = risk_matrix[["scenario_id", "scenario_weight"]].drop_duplicates()["scenario_weight"].sum()
if not np.isclose(scenario_weight_sum, 1.0, atol=1e-6):
    raise ValueError(f"Scenario weights should sum to 1. Current sum = {scenario_weight_sum}")

print("Risk matrix:", risk_matrix.shape)
print("Hybrid scenarios:", hybrid_scenarios.shape)
print("Full OOF predictions:", full_predictions.shape)
print("Unique reduced scenarios:", risk_matrix["scenario_id"].nunique())
print("Unique nodes:", risk_matrix["node_id"].nunique())
print("Scenario weight sum:", scenario_weight_sum)
print("Full OOF positive exposure count:", int(full_predictions["disrupted"].sum()))
print("Full OOF positive exposure ratio:", full_predictions["disrupted"].mean())

display_if_available(risk_matrix.head())

In [ ]:
# Cell 02: Build or load optimization node parameters and transportation arcs

NODE_PARAM_FILE = "optimization_node_parameters.csv"
ARC_FILE = "optimization_transport_arcs.csv"

node_param_path = find_file(NODE_PARAM_FILE, patterns=["*node*parameters*.csv"], required=False)  # optional; auto-created below if missing
arc_path = find_file(ARC_FILE, patterns=["*transport*arcs*.csv"], required=False)  # optional; auto-created below if missing


def haversine_km_scalar(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2.0) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2.0) ** 2
    return 2.0 * R * math.asin(math.sqrt(a))


def create_node_parameters():
    """Create the stylized 14-node supply-chain network used in the optimization model."""
    return pd.DataFrame([
        {"node_id": "N1",  "node_name": "Houston",        "role": "source", "fixed_cost": 0,    "capacity": 900,  "supply": 900,  "demand": 0,   "latitude": 29.7604, "longitude": -95.3698},
        {"node_id": "N2",  "node_name": "New Orleans",    "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 420, "latitude": 29.9511, "longitude": -90.0715},
        {"node_id": "N3",  "node_name": "Miami",          "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 520, "latitude": 25.7617, "longitude": -80.1918},
        {"node_id": "N4",  "node_name": "Orlando",        "role": "dc",     "fixed_cost": 1800, "capacity": 780,  "supply": 0,    "demand": 0,   "latitude": 28.5383, "longitude": -81.3792},
        {"node_id": "N5",  "node_name": "Atlanta",        "role": "dc",     "fixed_cost": 2100, "capacity": 1000, "supply": 0,    "demand": 0,   "latitude": 33.7490, "longitude": -84.3880},
        {"node_id": "N6",  "node_name": "Dallas",         "role": "source", "fixed_cost": 0,    "capacity": 1100, "supply": 1100, "demand": 0,   "latitude": 32.7767, "longitude": -96.7970},
        {"node_id": "N7",  "node_name": "Tampa",          "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 380, "latitude": 27.9506, "longitude": -82.4572},
        {"node_id": "N8",  "node_name": "Jacksonville",   "role": "dc",     "fixed_cost": 1600, "capacity": 700,  "supply": 0,    "demand": 0,   "latitude": 30.3322, "longitude": -81.6557},
        {"node_id": "N9",  "node_name": "Mobile",         "role": "source", "fixed_cost": 0,    "capacity": 650,  "supply": 650,  "demand": 0,   "latitude": 30.6954, "longitude": -88.0399},
        {"node_id": "N10", "node_name": "Savannah",       "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 300, "latitude": 32.0809, "longitude": -81.0912},
        {"node_id": "N11", "node_name": "Charleston",     "role": "market", "fixed_cost": 0,    "capacity": 0,    "supply": 0,    "demand": 280, "latitude": 32.7765, "longitude": -79.9311},
        {"node_id": "N12", "node_name": "Corpus Christi", "role": "source", "fixed_cost": 0,    "capacity": 700,  "supply": 700,  "demand": 0,   "latitude": 27.8006, "longitude": -97.3964},
        {"node_id": "N13", "node_name": "Baton Rouge",    "role": "dc",     "fixed_cost": 1300, "capacity": 620,  "supply": 0,    "demand": 0,   "latitude": 30.4515, "longitude": -91.1871},
        {"node_id": "N14", "node_name": "Charlotte",      "role": "dc",     "fixed_cost": 1700, "capacity": 760,  "supply": 0,    "demand": 0,   "latitude": 35.2271, "longitude": -80.8431},
    ])


def create_transport_arcs(node_params):
    sources_local = node_params[node_params["role"] == "source"]["node_id"].tolist()
    dcs_local = node_params[node_params["role"] == "dc"]["node_id"].tolist()
    markets_local = node_params[node_params["role"] == "market"]["node_id"].tolist()
    loc = node_params.set_index("node_id")[["latitude", "longitude"]].to_dict("index")
    rows = []
    cost_per_km = 0.015

    for i in sources_local:
        for j in dcs_local:
            dist = haversine_km_scalar(loc[i]["latitude"], loc[i]["longitude"], loc[j]["latitude"], loc[j]["longitude"])
            rows.append({"from_node": i, "to_node": j, "arc_type": "source_dc", "distance_km": dist, "unit_transport_cost": dist * cost_per_km})

    for j in dcs_local:
        for m in markets_local:
            dist = haversine_km_scalar(loc[j]["latitude"], loc[j]["longitude"], loc[m]["latitude"], loc[m]["longitude"])
            rows.append({"from_node": j, "to_node": m, "arc_type": "dc_market", "distance_km": dist, "unit_transport_cost": dist * cost_per_km})

    return pd.DataFrame(rows)

if node_param_path is None:
    print("Node-parameter file missing; creating optimization_node_parameters.csv from embedded study network.")
    node_params = create_node_parameters()
    save_csv(node_params, NODE_PARAM_FILE, also_root=True)
else:
    node_params = pd.read_csv(node_param_path)

if arc_path is None:
    print("Transport-arc file missing; creating optimization_transport_arcs.csv from embedded coordinates.")
    arcs_df = create_transport_arcs(node_params)
    save_csv(arcs_df, ARC_FILE, also_root=True)
else:
    arcs_df = pd.read_csv(arc_path)

required_node_cols = {"node_id", "node_name", "role", "fixed_cost", "capacity", "supply", "demand", "latitude", "longitude"}
required_arc_cols = {"from_node", "to_node", "arc_type", "distance_km", "unit_transport_cost"}
if required_node_cols - set(node_params.columns):
    raise ValueError(f"node_params missing columns: {sorted(required_node_cols - set(node_params.columns))}")
if required_arc_cols - set(arcs_df.columns):
    raise ValueError(f"arcs_df missing columns: {sorted(required_arc_cols - set(arcs_df.columns))}")

print("Node parameters:", node_params.shape)
print("Transport arcs:", arcs_df.shape)
print("Roles:")
print(node_params.groupby("role")["node_id"].apply(list))
print("Arc counts:")
print(arcs_df["arc_type"].value_counts())

In [ ]:
# Cell 03: Prepare optimization sets, dictionaries, and model parameters

sources = node_params[node_params["role"] == "source"]["node_id"].tolist()
dcs = node_params[node_params["role"] == "dc"]["node_id"].tolist()
markets = node_params[node_params["role"] == "market"]["node_id"].tolist()
all_nodes = node_params["node_id"].tolist()
scenarios = risk_matrix["scenario_id"].drop_duplicates().tolist()

scenario_weights = (
    risk_matrix[["scenario_id", "scenario_weight"]]
    .drop_duplicates()
    .set_index("scenario_id")["scenario_weight"]
    .astype(float)
    .to_dict()
)
weight_sum = sum(scenario_weights.values())
scenario_weights = {s: w / weight_sum for s, w in scenario_weights.items()}

p_is = risk_matrix.set_index(["scenario_id", "node_id"])["p_is"].astype(float).to_dict()
p_is_sensitivity = risk_matrix.set_index(["scenario_id", "node_id"])["p_is_sensitivity"].astype(float).to_dict()
p_rule_binary = risk_matrix.set_index(["scenario_id", "node_id"])["disrupted"].astype(float).to_dict()

fixed_cost = node_params.set_index("node_id")["fixed_cost"].astype(float).to_dict()
capacity = node_params.set_index("node_id")["capacity"].astype(float).to_dict()
supply = node_params.set_index("node_id")["supply"].astype(float).to_dict()
demand = node_params.set_index("node_id")["demand"].astype(float).to_dict()

cost_source_dc = (
    arcs_df[arcs_df["arc_type"] == "source_dc"]
    .set_index(["from_node", "to_node"])["unit_transport_cost"]
    .astype(float)
    .to_dict()
)
cost_dc_market = (
    arcs_df[arcs_df["arc_type"] == "dc_market"]
    .set_index(["from_node", "to_node"])["unit_transport_cost"]
    .astype(float)
    .to_dict()
)

TOTAL_DEMAND = sum(demand[m] for m in markets)

# Main model parameters used in the manuscript.
DEFAULT_ALPHA = 0.90
UNMET_PENALTY_PER_UNIT = 150.0
DISRUPTION_PENALTY_PER_CAPACITY = 4.0
MITIGATION_EFFECTIVENESS = 0.50
MIN_OPEN_DCS = 2

hardening_cost = {j: 0.45 * fixed_cost[j] + 250.0 for j in dcs}
disruption_penalty = {j: DISRUPTION_PENALTY_PER_CAPACITY * capacity[j] for j in dcs}

print("Sources:", sources)
print("Candidate DCs:", dcs)
print("Markets:", markets)
print("Scenarios:", len(scenarios))
print("Total demand:", TOTAL_DEMAND)
print("Scenario weight sum:", sum(scenario_weights.values()))
print("Default alpha:", DEFAULT_ALPHA)
print("Unmet penalty per unit:", UNMET_PENALTY_PER_UNIT)
print("Hardening effectiveness:", MITIGATION_EFFECTIVENESS)

In [ ]:
# Cell 04: Utility functions for design parsing, realized CVaR, and Pareto filtering


def parse_design_string(value):
    """Parse node strings such as 'N13,N4,N5' or 'N13;N4;N5' into a clean set."""
    if value is None:
        return set()
    if isinstance(value, float) and np.isnan(value):
        return set()
    if isinstance(value, (list, tuple, set)):
        return {str(x).strip() for x in value if str(x).strip()}
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "[]"}:
        return set()
    text = text.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    return {part.strip() for part in re.split(r"[,;]", text) if part.strip()}


def design_to_string(nodes):
    """Use the canonical DC order from the optimization model for stable output strings."""
    node_set = parse_design_string(nodes)
    ordered = [j for j in dcs if j in node_set]
    return ",".join(ordered)


def compute_weighted_cvar(loss_dict, weight_dict, alpha=DEFAULT_ALPHA):
    """Compute realized weighted CVaR_alpha from scenario-level loss values."""
    scenario_list = list(loss_dict.keys())
    losses = np.array([loss_dict[s] for s in scenario_list], dtype=float)
    weights = np.array([weight_dict[s] for s in scenario_list], dtype=float)
    weights = weights / weights.sum()

    # For discrete distributions, the minimizer of eta + E[(L-eta)+]/(1-alpha)
    # lies at one of the observed losses. This avoids relying on solver auxiliary variables.
    candidate_etas = np.unique(np.concatenate(([0.0], losses)))
    best_cvar = np.inf
    best_eta = None
    for eta in candidate_etas:
        value = eta + (1.0 / (1.0 - alpha)) * np.sum(weights * np.maximum(losses - eta, 0.0))
        if value < best_cvar - 1e-12:
            best_cvar = value
            best_eta = eta
    return float(best_cvar), float(best_eta)


def extract_nondominated(df, cost_col="total_cost", risk_col="realized_cvar_unmet_demand", tol=1e-6):
    """Extract nondominated rows for a two-objective minimization problem."""
    if df.empty:
        return df.copy()
    valid = df.dropna(subset=[cost_col, risk_col]).copy()
    if valid.empty:
        return valid

    valid["total_cost_round"] = valid[cost_col].round(4)
    valid["realized_cvar_round"] = valid[risk_col].round(4)
    subset_cols = ["total_cost_round", "realized_cvar_round"]
    for c in ["open_dcs", "hardened_dcs", "risk_method"]:
        if c in valid.columns:
            subset_cols.append(c)
    valid = valid.drop_duplicates(subset=subset_cols).copy()

    def dominated(row, df_check):
        others = df_check.drop(index=row.name)
        mask = (
            (others[cost_col] <= row[cost_col] + tol)
            & (others[risk_col] <= row[risk_col] + tol)
            & ((others[cost_col] < row[cost_col] - tol) | (others[risk_col] < row[risk_col] - tol))
        )
        return bool(mask.any())

    valid["is_dominated"] = valid.apply(lambda row: dominated(row, valid), axis=1)
    nd = valid[~valid["is_dominated"]].sort_values(by=[risk_col, cost_col], ascending=[False, True]).reset_index(drop=True)
    return nd


def build_result_row(method_name, point_id, cvar_limit, sol):
    return {
        "risk_method": method_name,
        "point_id": point_id,
        "cvar_limit": cvar_limit,
        "status": sol.get("status"),
        "total_cost": sol.get("total_cost", np.nan),
        "fixed_cost": sol.get("fixed_cost", np.nan),
        "hardening_cost": sol.get("hardening_cost", np.nan),
        "expected_transport_cost": sol.get("expected_transport_cost", np.nan),
        "expected_unmet_penalty": sol.get("expected_unmet_penalty", np.nan),
        "expected_disruption_penalty": sol.get("expected_disruption_penalty", np.nan),
        "realized_expected_unmet_demand": sol.get("realized_expected_unmet_demand", np.nan),
        "realized_cvar_unmet_demand": sol.get("realized_cvar_unmet_demand", np.nan),
        "realized_eta_cvar": sol.get("realized_eta_cvar", np.nan),
        "open_dcs": design_to_string(sol.get("open_dcs", [])),
        "hardened_dcs": design_to_string(sol.get("hardened_dcs", [])),
    }

print("Utility functions ready.")

In [ ]:
# Cell 05: Import Gurobi and define the two-stage stochastic CVaR SCND model

try:
    import gurobipy as gp
    from gurobipy import GRB
    GUROBI_AVAILABLE = True
    print("Gurobi imported successfully.")
    try:
        print("Gurobi version:", gp.gurobi.version())
    except Exception:
        pass
except Exception as e:
    gp = None
    GRB = None
    GUROBI_AVAILABLE = False
    print("Gurobi is not available in this Python environment.")
    print("Existing saved optimization outputs can still be loaded when FORCE_RECOMPUTE = False.")
    print("To recompute missing optimization results, install gurobipy and activate a valid Gurobi license.")
    print("Import error:", repr(e))


def solve_resilient_supply_chain_gurobi(
    objective="cost",
    cvar_limit=None,
    risk_dict=None,
    alpha=DEFAULT_ALPHA,
    hardening_effectiveness=MITIGATION_EFFECTIVENESS,
    unmet_penalty=UNMET_PENALTY_PER_UNIT,
    fixed_open_dcs=None,
    fixed_hardened_dcs=None,
    time_limit=300,
    mip_gap=1e-4,
    output_flag=0,
):
    """Solve the two-stage stochastic resilient supply-chain network design model.

    First-stage decisions:
        y_j: open candidate DC j.
        r_j: harden candidate DC j.

    Second-stage recourse:
        x_sij: source-to-DC flow under scenario s.
        z_sjm: DC-to-market flow under scenario s.
        unmet_sm: unmet demand at market m under scenario s.

    Objective:
        objective='cost' minimizes expected total cost.
        objective='cvar' minimizes CVaR of scenario-level unmet demand with a small cost tie-breaker.

    For common evaluation, fixed_open_dcs and fixed_hardened_dcs fix first-stage decisions
    while scenario-dependent recourse is re-optimized under an evaluation risk environment.
    """
    if not GUROBI_AVAILABLE:
        raise RuntimeError(
            "Gurobi is required to compute optimization results, but gurobipy is not available. "
            "Install/activate Gurobi or place existing result CSV files in the project folder with FORCE_RECOMPUTE = False."
        )

    if risk_dict is None:
        risk_dict = p_is

    fixed_open_dcs = parse_design_string(fixed_open_dcs) if fixed_open_dcs is not None else None
    fixed_hardened_dcs = parse_design_string(fixed_hardened_dcs) if fixed_hardened_dcs is not None else None

    model = gp.Model("Scenario_Based_Resilient_SCND")
    model.Params.OutputFlag = output_flag
    model.Params.TimeLimit = time_limit
    model.Params.MIPGap = mip_gap

    y = model.addVars(dcs, vtype=GRB.BINARY, name="open_dc")
    r = model.addVars(dcs, vtype=GRB.BINARY, name="harden_dc")

    x = model.addVars(scenarios, sources, dcs, lb=0.0, vtype=GRB.CONTINUOUS, name="flow_source_dc")
    z = model.addVars(scenarios, dcs, markets, lb=0.0, vtype=GRB.CONTINUOUS, name="flow_dc_market")
    unmet = model.addVars(scenarios, markets, lb=0.0, vtype=GRB.CONTINUOUS, name="unmet_demand")

    eta = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="eta_cvar")
    excess = model.addVars(scenarios, lb=0.0, vtype=GRB.CONTINUOUS, name="cvar_excess")

    if fixed_open_dcs is not None:
        for j in dcs:
            model.addConstr(y[j] == int(j in fixed_open_dcs), name=f"fix_open_{j}")
    if fixed_hardened_dcs is not None:
        for j in dcs:
            model.addConstr(r[j] == int(j in fixed_hardened_dcs), name=f"fix_harden_{j}")

    for j in dcs:
        model.addConstr(r[j] <= y[j], name=f"harden_only_if_open_{j}")
    model.addConstr(gp.quicksum(y[j] for j in dcs) >= MIN_OPEN_DCS, name="minimum_open_dcs")

    for s in scenarios:
        for i in sources:
            p_si = float(risk_dict.get((s, i), 0.0))
            model.addConstr(
                gp.quicksum(x[s, i, j] for j in dcs) <= supply[i] * (1.0 - p_si),
                name=f"source_capacity_{s}_{i}",
            )

    for s in scenarios:
        for j in dcs:
            p_sj = float(risk_dict.get((s, j), 0.0))
            effective_capacity = capacity[j] * ((1.0 - p_sj) * y[j] + hardening_effectiveness * p_sj * r[j])
            model.addConstr(
                gp.quicksum(z[s, j, m] for m in markets) <= effective_capacity,
                name=f"dc_capacity_{s}_{j}",
            )

    for s in scenarios:
        for j in dcs:
            model.addConstr(
                gp.quicksum(z[s, j, m] for m in markets) <= gp.quicksum(x[s, i, j] for i in sources),
                name=f"flow_conservation_{s}_{j}",
            )

    for s in scenarios:
        for m in markets:
            model.addConstr(
                gp.quicksum(z[s, j, m] for j in dcs) + unmet[s, m] >= demand[m],
                name=f"demand_satisfaction_{s}_{m}",
            )

    scenario_unmet_expr = {s: gp.quicksum(unmet[s, m] for m in markets) for s in scenarios}
    for s in scenarios:
        model.addConstr(excess[s] >= scenario_unmet_expr[s] - eta, name=f"cvar_excess_{s}")

    cvar_expr = eta + (1.0 / (1.0 - alpha)) * gp.quicksum(scenario_weights[s] * excess[s] for s in scenarios)
    if cvar_limit is not None:
        model.addConstr(cvar_expr <= float(cvar_limit), name="epsilon_cvar_limit")

    fixed_cost_expr = gp.quicksum(fixed_cost[j] * y[j] for j in dcs)
    hardening_cost_expr = gp.quicksum(hardening_cost[j] * r[j] for j in dcs)

    expected_transport_cost_expr = gp.quicksum(
        scenario_weights[s] * cost_source_dc[(i, j)] * x[s, i, j]
        for s in scenarios for i in sources for j in dcs
    ) + gp.quicksum(
        scenario_weights[s] * cost_dc_market[(j, m)] * z[s, j, m]
        for s in scenarios for j in dcs for m in markets
    )

    expected_unmet_penalty_expr = gp.quicksum(
        scenario_weights[s] * unmet_penalty * unmet[s, m]
        for s in scenarios for m in markets
    )

    expected_disruption_penalty_expr = gp.quicksum(
        scenario_weights[s]
        * disruption_penalty[j]
        * (float(risk_dict.get((s, j), 0.0)) * y[j] - hardening_effectiveness * float(risk_dict.get((s, j), 0.0)) * r[j])
        for s in scenarios for j in dcs
    )

    total_cost_expr = (
        fixed_cost_expr
        + hardening_cost_expr
        + expected_transport_cost_expr
        + expected_unmet_penalty_expr
        + expected_disruption_penalty_expr
    )

    expected_unmet_expr = gp.quicksum(scenario_weights[s] * scenario_unmet_expr[s] for s in scenarios)

    if objective == "cost":
        model.setObjective(total_cost_expr, GRB.MINIMIZE)
    elif objective == "cvar":
        model.setObjective(cvar_expr + 1e-6 * total_cost_expr, GRB.MINIMIZE)
    else:
        raise ValueError("objective must be either 'cost' or 'cvar'.")

    model.optimize()

    status_map = {
        GRB.OPTIMAL: "Optimal",
        GRB.TIME_LIMIT: "TimeLimit",
        GRB.INFEASIBLE: "Infeasible",
        GRB.INF_OR_UNBD: "InfOrUnbd",
        GRB.UNBOUNDED: "Unbounded",
    }
    status = status_map.get(model.status, f"Status_{model.status}")
    result = {"status": status, "objective": objective}

    if model.SolCount == 0:
        return result

    scenario_unmet = {s: float(sum(unmet[s, m].X for m in markets)) for s in scenarios}
    realized_expected_unmet = float(sum(scenario_weights[s] * scenario_unmet[s] for s in scenarios))
    realized_cvar, realized_eta = compute_weighted_cvar(scenario_unmet, scenario_weights, alpha=alpha)

    open_dcs = [j for j in dcs if y[j].X > 0.5]
    hardened_dcs = [j for j in dcs if r[j].X > 0.5]

    result.update({
        "total_cost": float(total_cost_expr.getValue()),
        "fixed_cost": float(fixed_cost_expr.getValue()),
        "hardening_cost": float(hardening_cost_expr.getValue()),
        "expected_transport_cost": float(expected_transport_cost_expr.getValue()),
        "expected_unmet_penalty": float(expected_unmet_penalty_expr.getValue()),
        "expected_disruption_penalty": float(expected_disruption_penalty_expr.getValue()),
        "expected_unmet_demand": float(expected_unmet_expr.getValue()),
        "internal_cvar_unmet_demand": float(cvar_expr.getValue()),
        "eta_cvar": float(eta.X),
        "realized_expected_unmet_demand": realized_expected_unmet,
        "realized_cvar_unmet_demand": float(realized_cvar),
        "realized_eta_cvar": float(realized_eta),
        "open_dcs": open_dcs,
        "hardened_dcs": hardened_dcs,
        "open_dcs_str": design_to_string(open_dcs),
        "hardened_dcs_str": design_to_string(hardened_dcs),
        "scenario_unmet": scenario_unmet,
    })

    return result

print("Gurobi SCND model function ready.")

In [ ]:
# Cell 06: Construct risk-input strategies for optimization comparison

# 1. Proposed ML Dynamic risk: calibrated scenario-node probabilities from Notebook 01.
risk_ml_dynamic = p_is.copy()

# 2. Fixed baseline: smoothed global historical exposure rate.
global_positive = float(full_predictions["disrupted"].sum())
global_total = float(full_predictions.shape[0])
fixed_probability = (global_positive + 1.0) / (global_total + 2.0)
risk_fixed = {(s, i): fixed_probability for s in scenarios for i in all_nodes}

# 3. Historical baseline: Laplace-smoothed node-level exposure frequency.
node_frequency = full_predictions.groupby("node_id")["disrupted"].agg(["sum", "count"]).reset_index()
node_frequency["historical_probability"] = (node_frequency["sum"] + 1.0) / (node_frequency["count"] + 2.0)
node_hist_prob = node_frequency.set_index("node_id")["historical_probability"].to_dict()
risk_historical = {(s, i): float(node_hist_prob.get(i, fixed_probability)) for s in scenarios for i in all_nodes}

# 4. Rule-based baseline: binary exposure label in each reduced scenario.
risk_rule_based = p_rule_binary.copy()

risk_methods = {
    "Fixed": risk_fixed,
    "Historical": risk_historical,
    "Rule-based": risk_rule_based,
    "ML Dynamic": risk_ml_dynamic,
}

risk_method_summary_df = pd.DataFrame([
    {
        "risk_method": name,
        "mean_risk": float(np.mean(list(risk_dict.values()))),
        "std_risk": float(np.std(list(risk_dict.values()))),
        "min_risk": float(np.min(list(risk_dict.values()))),
        "max_risk": float(np.max(list(risk_dict.values()))),
        "nonzero_ratio": float(np.mean(np.array(list(risk_dict.values())) > 0)),
    }
    for name, risk_dict in risk_methods.items()
])
save_csv(risk_method_summary_df, "risk_method_summary.csv", also_root=True)

print("Fixed smoothed probability:", fixed_probability)
print("Node historical probabilities:")
display_if_available(node_frequency.sort_values("historical_probability", ascending=False))
print("Risk-method summary:")
display_if_available(risk_method_summary_df)

In [ ]:
# Cell 07: Define Pareto-frontier generation using realized CVaR


def generate_pareto_for_risk_method(
    method_name,
    risk_dict,
    alpha=DEFAULT_ALPHA,
    hardening_effectiveness=MITIGATION_EFFECTIVENESS,
    unmet_penalty=UNMET_PENALTY_PER_UNIT,
    num_points=20,
    time_limit=300,
    mip_gap=1e-4,
):
    """Generate a realized cost-CVaR Pareto frontier for one risk-input strategy."""
    print("=" * 100)
    print(f"Generating Pareto frontier for risk method: {method_name}")

    min_cost_sol = solve_resilient_supply_chain_gurobi(
        objective="cost",
        risk_dict=risk_dict,
        alpha=alpha,
        hardening_effectiveness=hardening_effectiveness,
        unmet_penalty=unmet_penalty,
        time_limit=time_limit,
        mip_gap=mip_gap,
        output_flag=0,
    )
    min_cvar_sol = solve_resilient_supply_chain_gurobi(
        objective="cvar",
        risk_dict=risk_dict,
        alpha=alpha,
        hardening_effectiveness=hardening_effectiveness,
        unmet_penalty=unmet_penalty,
        time_limit=time_limit,
        mip_gap=mip_gap,
        output_flag=0,
    )

    if min_cost_sol.get("status") not in {"Optimal", "TimeLimit"} or min_cvar_sol.get("status") not in {"Optimal", "TimeLimit"}:
        raise RuntimeError(
            f"Anchor solve failed for {method_name}: "
            f"min_cost={min_cost_sol.get('status')}, min_cvar={min_cvar_sol.get('status')}"
        )

    cvar_high = float(min_cost_sol["realized_cvar_unmet_demand"])
    cvar_low = float(min_cvar_sol["realized_cvar_unmet_demand"])

    print("Min-cost realized CVaR:", cvar_high)
    print("Min-CVaR realized CVaR:", cvar_low)

    if abs(cvar_high - cvar_low) < 1e-6:
        epsilon_values = np.array([cvar_high])
    else:
        epsilon_values = np.linspace(cvar_high, cvar_low, num_points)

    rows = []
    for idx, eps in enumerate(epsilon_values, start=1):
        print(f"  Solving {method_name} point {idx}/{len(epsilon_values)} | CVaR limit = {eps:.6f}")
        sol = solve_resilient_supply_chain_gurobi(
            objective="cost",
            cvar_limit=float(eps),
            risk_dict=risk_dict,
            alpha=alpha,
            hardening_effectiveness=hardening_effectiveness,
            unmet_penalty=unmet_penalty,
            time_limit=time_limit,
            mip_gap=mip_gap,
            output_flag=0,
        )
        rows.append(build_result_row(method_name, idx, eps, sol))

    raw_df = pd.DataFrame(rows)
    valid_df = raw_df[raw_df["status"].isin(["Optimal", "TimeLimit"])].copy()
    valid_df = valid_df.dropna(subset=["total_cost", "realized_cvar_unmet_demand"])
    if valid_df.empty:
        return raw_df, valid_df

    nd_df = extract_nondominated(valid_df, cost_col="total_cost", risk_col="realized_cvar_unmet_demand")
    return raw_df, nd_df

print("Pareto generation function ready.")

In [ ]:
# Cell 08: Solve ML Dynamic Pareto frontier and write corrected anchor summary

# Important correction:
# The pure "minimize CVaR" Gurobi anchor can return any zero-CVaR solution when many
# equivalent CVaR-optimal solutions exist, and it does not necessarily minimize cost
# within the zero-CVaR set. For reporting, the min-CVaR anchor must therefore be the
# cost-minimal solution among the Pareto solutions that attain the minimum realized CVaR.
# This keeps the anchor table consistent with the realized-CVaR Pareto frontier used in
# the paper and in common evaluation.

ml_dynamic_raw = maybe_load_csv("gurobi_pareto_frontier_ml_dynamic_risk_realized_cvar.csv")
ml_dynamic_nd = maybe_load_csv("gurobi_pareto_frontier_ml_dynamic_risk_nondominated.csv")

if ml_dynamic_raw is None or ml_dynamic_nd is None:
    ml_dynamic_raw, ml_dynamic_nd = generate_pareto_for_risk_method(
        "ML Dynamic",
        risk_ml_dynamic,
        alpha=DEFAULT_ALPHA,
        num_points=20,
        time_limit=300,
        mip_gap=1e-4,
    )
    save_csv(ml_dynamic_raw, "gurobi_pareto_frontier_ml_dynamic_risk_realized_cvar.csv", also_root=True)
    save_csv(ml_dynamic_nd, "gurobi_pareto_frontier_ml_dynamic_risk_nondominated.csv", also_root=True)

required_pareto_cols = [
    "status", "total_cost", "realized_expected_unmet_demand",
    "realized_cvar_unmet_demand", "open_dcs", "hardened_dcs"
]

missing_pareto_cols = [col for col in required_pareto_cols if col not in ml_dynamic_nd.columns]
if missing_pareto_cols:
    raise ValueError(f"ML Dynamic nondominated Pareto file is missing columns: {missing_pareto_cols}")

valid_pareto = ml_dynamic_nd[ml_dynamic_nd["status"].isin(["Optimal", "TimeLimit"])].copy()
valid_pareto = valid_pareto.dropna(subset=["total_cost", "realized_cvar_unmet_demand"])

if valid_pareto.empty:
    raise RuntimeError("No valid ML Dynamic nondominated Pareto solutions are available for anchor extraction.")

# Min-cost anchor: lowest realized total cost on the reported Pareto frontier.
min_cost_row = (
    valid_pareto
    .sort_values(by=["total_cost", "realized_cvar_unmet_demand"], ascending=[True, True])
    .iloc[0]
)

# Min-CVaR anchor: among the solutions attaining minimum realized CVaR, select the
# lowest-cost one. This avoids reporting a cost-inefficient zero-CVaR solution.
min_cvar_value = float(valid_pareto["realized_cvar_unmet_demand"].min())
CV_TOL = 1e-6

min_cvar_candidates = valid_pareto[
    valid_pareto["realized_cvar_unmet_demand"] <= min_cvar_value + CV_TOL
].copy()

min_cvar_row = (
    min_cvar_candidates
    .sort_values(by=["total_cost", "realized_cvar_unmet_demand"], ascending=[True, True])
    .iloc[0]
)

anchor_df = pd.DataFrame([
    {
        "solution": "min_cost",
        "source": "cost-minimal nondominated Pareto solution",
        "source_point_id": min_cost_row.get("point_id", np.nan),
        "status": min_cost_row.get("status"),
        "total_cost": float(min_cost_row["total_cost"]),
        "expected_unmet_demand": float(min_cost_row["realized_expected_unmet_demand"]),
        "realized_cvar_unmet_demand": float(min_cost_row["realized_cvar_unmet_demand"]),
        "open_dcs": min_cost_row.get("open_dcs", ""),
        "hardened_dcs": min_cost_row.get("hardened_dcs", ""),
    },
    {
        "solution": "min_cvar",
        "source": "cost-minimal solution among minimum-realized-CVaR Pareto solutions",
        "source_point_id": min_cvar_row.get("point_id", np.nan),
        "status": min_cvar_row.get("status"),
        "total_cost": float(min_cvar_row["total_cost"]),
        "expected_unmet_demand": float(min_cvar_row["realized_expected_unmet_demand"]),
        "realized_cvar_unmet_demand": float(min_cvar_row["realized_cvar_unmet_demand"]),
        "open_dcs": min_cvar_row.get("open_dcs", ""),
        "hardened_dcs": min_cvar_row.get("hardened_dcs", ""),
    },
])

save_csv(anchor_df, "gurobi_optimization_anchor_solutions.csv", also_root=True)

print("Corrected anchor solutions:")
display_if_available(anchor_df)

print("ML Dynamic nondominated Pareto solutions:")
display_if_available(ml_dynamic_nd)

In [ ]:
# Cell 09: Plot ML Dynamic Pareto frontier and summarize design transitions

import matplotlib.pyplot as plt

valid_ml = ml_dynamic_raw[ml_dynamic_raw["status"].isin(["Optimal", "TimeLimit"])].copy()
valid_ml = valid_ml.dropna(subset=["total_cost", "realized_cvar_unmet_demand"])

plt.figure(figsize=(7, 5))
plt.scatter(
    valid_ml["realized_cvar_unmet_demand"],
    valid_ml["total_cost"],
    marker="o",
    label="Generated solutions",
)
plt.plot(
    ml_dynamic_nd.sort_values("realized_cvar_unmet_demand")["realized_cvar_unmet_demand"],
    ml_dynamic_nd.sort_values("realized_cvar_unmet_demand")["total_cost"],
    marker="o",
    linestyle="--",
    label="Nondominated frontier",
)
plt.xlabel("Realized CVaR of unmet demand")
plt.ylabel("Total expected cost")
plt.title("ML Dynamic Risk: Cost-CVaR Pareto Frontier")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "corrected_pareto_frontier_ml_dynamic_risk.png", dpi=FIGURE_DPI)
plt.savefig(BASE_DIR / "corrected_pareto_frontier_ml_dynamic_risk.png", dpi=FIGURE_DPI)
plt.show()

corrected_design_summary = ml_dynamic_nd[[
    "point_id", "total_cost", "realized_expected_unmet_demand", "realized_cvar_unmet_demand", "open_dcs", "hardened_dcs"
]].copy()
corrected_design_summary["number_open_dcs"] = corrected_design_summary["open_dcs"].apply(lambda x: len(parse_design_string(x)))
corrected_design_summary["number_hardened_dcs"] = corrected_design_summary["hardened_dcs"].apply(lambda x: len(parse_design_string(x)))
save_csv(corrected_design_summary, "corrected_pareto_design_transition_summary.csv", also_root=True)

display_if_available(corrected_design_summary)

In [ ]:
# Cell 10: Generate Pareto frontiers for all risk-input methods

raw_all = maybe_load_csv("raw_pareto_comparison_all_risk_methods.csv")
nd_all = maybe_load_csv("nondominated_pareto_comparison_all_risk_methods.csv")

if raw_all is None or nd_all is None:
    raw_list = []
    nd_list = []
    for method_name, risk_dict in risk_methods.items():
        raw_df, nd_df = generate_pareto_for_risk_method(
            method_name,
            risk_dict,
            alpha=DEFAULT_ALPHA,
            num_points=20,
            time_limit=300,
            mip_gap=1e-4,
        )
        raw_list.append(raw_df)
        nd_list.append(nd_df)
    raw_all = pd.concat(raw_list, ignore_index=True)
    nd_all = pd.concat(nd_list, ignore_index=True)
    save_csv(raw_all, "raw_pareto_comparison_all_risk_methods.csv", also_root=True)
    save_csv(nd_all, "nondominated_pareto_comparison_all_risk_methods.csv", also_root=True)

comparison_diagnostic = (
    nd_all.groupby("risk_method")
    .agg(
        nondominated_points=("point_id", "count"),
        min_total_cost=("total_cost", "min"),
        max_total_cost=("total_cost", "max"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
    )
    .reset_index()
)
save_csv(comparison_diagnostic, "pareto_comparison_diagnostic.csv", also_root=True)

display_if_available(comparison_diagnostic)

In [ ]:
# Cell 11: Extract representative designs and create candidate design set for common evaluation

representative_df = maybe_load_csv("representative_designs_by_risk_method.csv")

def choose_representative_designs(nd_df):
    rows = []
    for method_name, temp in nd_df.groupby("risk_method"):
        temp = temp.dropna(subset=["total_cost", "realized_cvar_unmet_demand"]).copy()
        if temp.empty:
            continue
        lowest_cost = temp.sort_values("total_cost").iloc[0]
        highest_resilience = temp.sort_values("realized_cvar_unmet_demand").iloc[0]

        temp_sorted = temp.sort_values("realized_cvar_unmet_demand", ascending=False).copy()
        if len(temp_sorted) <= 2:
            knee = lowest_cost
        else:
            c = temp_sorted["total_cost"].to_numpy(dtype=float)
            r = temp_sorted["realized_cvar_unmet_demand"].to_numpy(dtype=float)
            c_norm = (c - c.min()) / (c.max() - c.min() + 1e-12)
            r_norm = (r - r.min()) / (r.max() - r.min() + 1e-12)
            # Maximize distance from the straight line between endpoints in normalized objective space.
            line = np.linspace(r_norm[0], r_norm[-1], len(r_norm))
            knee_idx = int(np.argmax(np.abs(r_norm - line) + 0.1 * c_norm))
            knee = temp_sorted.iloc[knee_idx]

        for design_type, row in [
            ("lowest_cost", lowest_cost),
            ("knee_solution", knee),
            ("highest_resilience", highest_resilience),
        ]:
            rows.append({
                "risk_method": method_name,
                "design_type": design_type,
                "total_cost": row["total_cost"],
                "realized_expected_unmet_demand": row["realized_expected_unmet_demand"],
                "realized_cvar_unmet_demand": row["realized_cvar_unmet_demand"],
                "fixed_cost": row.get("fixed_cost", np.nan),
                "hardening_cost": row.get("hardening_cost", np.nan),
                "expected_transport_cost": row.get("expected_transport_cost", np.nan),
                "expected_unmet_penalty": row.get("expected_unmet_penalty", np.nan),
                "expected_disruption_penalty": row.get("expected_disruption_penalty", np.nan),
                "open_dcs": row["open_dcs"],
                "hardened_dcs": row["hardened_dcs"],
            })
    return pd.DataFrame(rows)

if representative_df is None:
    representative_df = choose_representative_designs(nd_all)
    save_csv(representative_df, "representative_designs_by_risk_method.csv", also_root=True)

# Common evaluation uses every nondominated design from the all-method Pareto comparison.
designs_to_evaluate = maybe_load_csv("designs_to_common_evaluate.csv")
if designs_to_evaluate is None:
    design_rows = []
    for _, row in nd_all.iterrows():
        method = row["risk_method"]
        point_id = int(row["point_id"])
        design_rows.append({
            "design_id": f"{method}_P{point_id}",
            "risk_method": method,
            "source_point_id": point_id,
            "internal_total_cost": row["total_cost"],
            "internal_realized_cvar": row["realized_cvar_unmet_demand"],
            "open_dcs": row["open_dcs"],
            "hardened_dcs": row["hardened_dcs"],
            "number_open_dcs": len(parse_design_string(row["open_dcs"])),
            "number_hardened_dcs": len(parse_design_string(row["hardened_dcs"])),
        })
    designs_to_evaluate = pd.DataFrame(design_rows)
    save_csv(designs_to_evaluate, "designs_to_common_evaluate.csv", also_root=True)

# Defensive normalization.
for col in ["open_dcs", "hardened_dcs"]:
    designs_to_evaluate[col] = designs_to_evaluate[col].apply(design_to_string)

display_if_available(representative_df)
print("Candidate designs for common evaluation:", designs_to_evaluate.shape)
display_if_available(designs_to_evaluate)

In [ ]:
# Cell 12: Define common-evaluation function for fixed first-stage designs


def evaluate_fixed_design_gurobi(
    open_dcs_set,
    hardened_dcs_set,
    evaluation_risk_dict,
    alpha=DEFAULT_ALPHA,
    hardening_effectiveness=MITIGATION_EFFECTIVENESS,
    unmet_penalty=UNMET_PENALTY_PER_UNIT,
    time_limit=300,
    mip_gap=1e-4,
    output_flag=0,
):
    """Fix a first-stage design and re-optimize recourse under a common risk environment."""
    sol = solve_resilient_supply_chain_gurobi(
        objective="cost",
        risk_dict=evaluation_risk_dict,
        alpha=alpha,
        hardening_effectiveness=hardening_effectiveness,
        unmet_penalty=unmet_penalty,
        fixed_open_dcs=open_dcs_set,
        fixed_hardened_dcs=hardened_dcs_set,
        time_limit=time_limit,
        mip_gap=mip_gap,
        output_flag=output_flag,
    )

    return {
        "status": sol.get("status"),
        "evaluated_total_cost": sol.get("total_cost", np.nan),
        "evaluated_fixed_cost": sol.get("fixed_cost", np.nan),
        "evaluated_hardening_cost": sol.get("hardening_cost", np.nan),
        "evaluated_transport_cost": sol.get("expected_transport_cost", np.nan),
        "evaluated_unmet_penalty": sol.get("expected_unmet_penalty", np.nan),
        "evaluated_disruption_penalty": sol.get("expected_disruption_penalty", np.nan),
        "evaluated_expected_unmet_demand": sol.get("realized_expected_unmet_demand", np.nan),
        "evaluated_cvar_unmet_demand": sol.get("realized_cvar_unmet_demand", np.nan),
        "evaluated_eta_cvar": sol.get("realized_eta_cvar", np.nan),
    }

print("Common-evaluation function ready.")

In [ ]:
# Cell 13: Evaluate all candidate designs under common calibrated ML Dynamic risk

common_evaluation_df = maybe_load_csv("common_evaluation_under_ml_dynamic_risk.csv")

if common_evaluation_df is None:
    common_eval_rows = []
    for _, row in designs_to_evaluate.iterrows():
        print(f"Evaluating {row['design_id']} under common ML Dynamic risk")
        open_set = parse_design_string(row["open_dcs"])
        harden_set = parse_design_string(row["hardened_dcs"])
        eval_result = evaluate_fixed_design_gurobi(
            open_dcs_set=open_set,
            hardened_dcs_set=harden_set,
            evaluation_risk_dict=risk_ml_dynamic,
            alpha=DEFAULT_ALPHA,
            time_limit=300,
            mip_gap=1e-4,
            output_flag=0,
        )
        common_eval_rows.append({
            "design_id": row["design_id"],
            "risk_method": row["risk_method"],
            "source_point_id": row["source_point_id"],
            "open_dcs": design_to_string(open_set),
            "hardened_dcs": design_to_string(harden_set),
            "number_open_dcs": len(open_set),
            "number_hardened_dcs": len(harden_set),
            "internal_total_cost": row["internal_total_cost"],
            "internal_realized_cvar": row["internal_realized_cvar"],
            **eval_result,
        })
    common_evaluation_df = pd.DataFrame(common_eval_rows)
    save_csv(common_evaluation_df, "common_evaluation_under_ml_dynamic_risk.csv", also_root=True)

common_valid = common_evaluation_df[common_evaluation_df["status"].isin(["Optimal", "TimeLimit"])].copy()
renamed_for_nd = common_valid.rename(
    columns={
        "evaluated_total_cost": "total_cost",
        "evaluated_cvar_unmet_demand": "realized_cvar_unmet_demand",
    }
)
common_nondominated = extract_nondominated(
    renamed_for_nd,
    cost_col="total_cost",
    risk_col="realized_cvar_unmet_demand",
).rename(
    columns={
        "total_cost": "evaluated_total_cost",
        "realized_cvar_unmet_demand": "evaluated_cvar_unmet_demand",
    }
)
save_csv(common_nondominated, "common_evaluation_nondominated_designs_under_ml_dynamic.csv", also_root=True)

print("Common evaluation results:")
display_if_available(common_evaluation_df, max_rows=30)

In [ ]:
# Cell 14: Summarize common evaluation and quantify improvements

common_eval_summary = (
    common_evaluation_df
    .groupby("risk_method")
    .agg(
        design_count=("design_id", "count"),
        min_cost=("evaluated_total_cost", "min"),
        max_cost=("evaluated_total_cost", "max"),
        min_cvar=("evaluated_cvar_unmet_demand", "min"),
        max_cvar=("evaluated_cvar_unmet_demand", "max"),
        min_expected_unmet=("evaluated_expected_unmet_demand", "min"),
        max_expected_unmet=("evaluated_expected_unmet_demand", "max"),
    )
    .reset_index()
)
save_csv(common_eval_summary, "common_evaluation_summary_by_risk_method.csv", also_root=True)

baseline_methods = ["Fixed", "Historical", "Rule-based"]
ml_designs = common_evaluation_df[common_evaluation_df["risk_method"] == "ML Dynamic"].copy()

improvement_rows = []
for baseline_method in baseline_methods:
    baseline_candidates = common_evaluation_df[common_evaluation_df["risk_method"] == baseline_method].copy()
    if baseline_candidates.empty or ml_designs.empty:
        continue
    baseline = baseline_candidates.sort_values("evaluated_total_cost").iloc[0]

    # For Fixed/Historical, compare against the lowest-cost ML design.
    # For Rule-based, compare against the lowest-cost ML design with lower evaluated CVaR.
    if baseline_method in ["Fixed", "Historical"]:
        chosen_ml = ml_designs.sort_values("evaluated_total_cost").iloc[0]
    else:
        feasible = ml_designs[ml_designs["evaluated_cvar_unmet_demand"] < baseline["evaluated_cvar_unmet_demand"]].copy()
        chosen_ml = feasible.sort_values("evaluated_total_cost").iloc[0] if not feasible.empty else ml_designs.sort_values("evaluated_cvar_unmet_demand").iloc[0]

    improvement_rows.append({
        "baseline_method": baseline_method,
        "baseline_design_id": baseline["design_id"],
        "baseline_cost": baseline["evaluated_total_cost"],
        "baseline_cvar": baseline["evaluated_cvar_unmet_demand"],
        "baseline_expected_unmet": baseline["evaluated_expected_unmet_demand"],
        "baseline_open_dcs": baseline["open_dcs"],
        "baseline_hardened_dcs": baseline["hardened_dcs"],
        "comparison_ml_design_id": chosen_ml["design_id"],
        "ml_cost": chosen_ml["evaluated_total_cost"],
        "ml_cvar": chosen_ml["evaluated_cvar_unmet_demand"],
        "ml_expected_unmet": chosen_ml["evaluated_expected_unmet_demand"],
        "ml_open_dcs": chosen_ml["open_dcs"],
        "ml_hardened_dcs": chosen_ml["hardened_dcs"],
        "absolute_cost_change": chosen_ml["evaluated_total_cost"] - baseline["evaluated_total_cost"],
        "absolute_cvar_reduction": baseline["evaluated_cvar_unmet_demand"] - chosen_ml["evaluated_cvar_unmet_demand"],
        "absolute_expected_unmet_reduction": baseline["evaluated_expected_unmet_demand"] - chosen_ml["evaluated_expected_unmet_demand"],
        "percent_cost_change": 100.0 * (chosen_ml["evaluated_total_cost"] - baseline["evaluated_total_cost"]) / baseline["evaluated_total_cost"],
        "percent_cvar_reduction": 100.0 * (baseline["evaluated_cvar_unmet_demand"] - chosen_ml["evaluated_cvar_unmet_demand"]) / baseline["evaluated_cvar_unmet_demand"] if baseline["evaluated_cvar_unmet_demand"] > 1e-9 else np.nan,
    })

improvement_summary = pd.DataFrame(improvement_rows)
save_csv(improvement_summary, "common_evaluation_improvement_summary.csv", also_root=True)

# Publication table aliases.
save_csv(improvement_summary, "table_main_common_evaluation_improvement.csv", also_root=True)

representative_table = common_evaluation_df[[
    "design_id", "risk_method", "open_dcs", "hardened_dcs",
    "evaluated_total_cost", "evaluated_expected_unmet_demand", "evaluated_cvar_unmet_demand"
]].copy()
save_csv(representative_table, "table_representative_designs_common_evaluation.csv", also_root=True)

print("Common-evaluation summary:")
display_if_available(common_eval_summary)
print("Improvement summary:")
display_if_available(improvement_summary)

In [ ]:
# Cell 15: Plot common-evaluation results for manuscript figures

plt.figure(figsize=(8, 6))
markers = {"Fixed": "o", "Historical": "s", "Rule-based": "^", "ML Dynamic": "D"}

for method_name, temp in common_evaluation_df.groupby("risk_method"):
    plt.scatter(
        temp["evaluated_cvar_unmet_demand"],
        temp["evaluated_total_cost"],
        marker=markers.get(method_name, "o"),
        s=90 if method_name != "ML Dynamic" else 115,
        edgecolor="black",
        alpha=0.85,
        label=method_name,
    )

frontier = common_nondominated.sort_values("evaluated_cvar_unmet_demand")
plt.plot(
    frontier["evaluated_cvar_unmet_demand"],
    frontier["evaluated_total_cost"],
    linestyle="--",
    marker="o",
    linewidth=1.6,
    label="Common-evaluation nondominated frontier",
)

for label in ["ML Dynamic_P1", "ML Dynamic_P2", "ML Dynamic_P20"]:
    temp = common_evaluation_df[common_evaluation_df["design_id"] == label]
    if not temp.empty:
        row = temp.iloc[0]
        plt.annotate(label, (row["evaluated_cvar_unmet_demand"], row["evaluated_total_cost"]), xytext=(8, 7), textcoords="offset points")

plt.xlabel("Evaluated CVaR of unmet demand under common ML Dynamic risk")
plt.ylabel("Evaluated total expected cost")
plt.title("Common Evaluation of Risk-Input-Driven Supply Chain Designs")
plt.grid(True, alpha=0.30)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "publication_common_evaluation_designs_clean_v2.png", dpi=FIGURE_DPI)
plt.savefig(BASE_DIR / "publication_common_evaluation_designs_clean_v2.png", dpi=FIGURE_DPI)
plt.show()

plt.figure(figsize=(7, 5))
bar_df = improvement_summary.copy()
plt.bar(bar_df["baseline_method"], bar_df["percent_cvar_reduction"])
for idx, row in bar_df.iterrows():
    val = row["percent_cvar_reduction"]
    if pd.notna(val):
        plt.text(idx, val + 1.0, f"{val:.1f}%", ha="center")
plt.ylabel("CVaR reduction by selected ML Dynamic design (%)")
plt.xlabel("Baseline risk-input method")
plt.title("Tail-Risk Reduction under Common Probabilistic Evaluation")
plt.ylim(0, max(10, float(bar_df["percent_cvar_reduction"].max()) * 1.25))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bar_cvar_reduction_over_baselines.png", dpi=FIGURE_DPI)
plt.savefig(BASE_DIR / "bar_cvar_reduction_over_baselines.png", dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Cell 16: Robustness under multiple common-evaluation environments

# ML calibrated: final calibrated p_is.
# ML uncalibrated: p_is_sensitivity from Notebook 01.
# Rule binary: original exposure indicator.
evaluation_environments = {
    "ML_calibrated": risk_ml_dynamic,
    "ML_uncalibrated": p_is_sensitivity,
    "Rule_binary": p_rule_binary,
}

evaluation_environment_summary = pd.DataFrame([
    {
        "evaluation_environment": env_name,
        "mean_risk": float(np.mean(list(env_risk.values()))),
        "std_risk": float(np.std(list(env_risk.values()))),
        "min_risk": float(np.min(list(env_risk.values()))),
        "max_risk": float(np.max(list(env_risk.values()))),
        "nonzero_ratio": float(np.mean(np.array(list(env_risk.values())) > 0)),
    }
    for env_name, env_risk in evaluation_environments.items()
])
save_csv(evaluation_environment_summary, "evaluation_environment_summary.csv", also_root=True)

multi_environment_evaluation_df = maybe_load_csv("multi_environment_common_evaluation.csv")

if multi_environment_evaluation_df is None:
    multi_env_rows = []
    for env_name, env_risk in evaluation_environments.items():
        print("=" * 100)
        print("Evaluation environment:", env_name)
        for _, row in designs_to_evaluate.iterrows():
            print(f"Evaluating {row['design_id']} under {env_name}")
            open_set = parse_design_string(row["open_dcs"])
            harden_set = parse_design_string(row["hardened_dcs"])
            eval_result = evaluate_fixed_design_gurobi(
                open_dcs_set=open_set,
                hardened_dcs_set=harden_set,
                evaluation_risk_dict=env_risk,
                alpha=DEFAULT_ALPHA,
                time_limit=300,
                mip_gap=1e-4,
                output_flag=0,
            )
            multi_env_rows.append({
                "evaluation_environment": env_name,
                "design_id": row["design_id"],
                "risk_method": row["risk_method"],
                "source_point_id": row["source_point_id"],
                "open_dcs": design_to_string(open_set),
                "hardened_dcs": design_to_string(harden_set),
                "number_open_dcs": len(open_set),
                "number_hardened_dcs": len(harden_set),
                **eval_result,
            })
    multi_environment_evaluation_df = pd.DataFrame(multi_env_rows)
    save_csv(multi_environment_evaluation_df, "multi_environment_common_evaluation.csv", also_root=True)

multi_environment_summary = (
    multi_environment_evaluation_df
    .groupby(["evaluation_environment", "risk_method"])
    .agg(
        design_count=("design_id", "count"),
        min_cost=("evaluated_total_cost", "min"),
        min_cvar=("evaluated_cvar_unmet_demand", "min"),
        min_expected_unmet=("evaluated_expected_unmet_demand", "min"),
    )
    .reset_index()
)
save_csv(multi_environment_summary, "multi_environment_common_evaluation_summary.csv", also_root=True)

best_designs_rows = []
for (env_name, method_name), temp in multi_environment_evaluation_df.groupby(["evaluation_environment", "risk_method"]):
    best = temp.sort_values(["evaluated_cvar_unmet_demand", "evaluated_total_cost"]).iloc[0]
    best_designs_rows.append(best.to_dict())
best_designs_multi_environment = pd.DataFrame(best_designs_rows)
save_csv(best_designs_multi_environment, "best_designs_multi_environment.csv", also_root=True)

display_if_available(multi_environment_summary, max_rows=30)

In [ ]:
# Cell 17: Sensitivity analysis settings

ALPHA_VALUES = [0.80, 0.90, 0.95]
HARDENING_EFFECTIVENESS_VALUES = [0.30, 0.50, 0.70]
NUM_POINTS_SENSITIVITY = 12

print("Sensitivity alpha values:", ALPHA_VALUES)
print("Hardening effectiveness values:", HARDENING_EFFECTIVENESS_VALUES)
print("Epsilon points per sensitivity setting:", NUM_POINTS_SENSITIVITY)

In [ ]:
# Cell 18: Sensitivity analysis on CVaR confidence level alpha

alpha_sensitivity_raw_df = maybe_load_csv("sensitivity_alpha_raw_pareto.csv")
alpha_sensitivity_nd_df = maybe_load_csv("sensitivity_alpha_nondominated_pareto.csv")

if alpha_sensitivity_raw_df is None or alpha_sensitivity_nd_df is None:
    raw_list = []
    nd_list = []
    for alpha_value in ALPHA_VALUES:
        method_label = f"ML Dynamic_alpha_{alpha_value}"
        print("=" * 100)
        print("Running alpha sensitivity:", alpha_value)
        raw_df, nd_df = generate_pareto_for_risk_method(
            method_name=method_label,
            risk_dict=risk_ml_dynamic,
            alpha=alpha_value,
            hardening_effectiveness=MITIGATION_EFFECTIVENESS,
            num_points=NUM_POINTS_SENSITIVITY,
            time_limit=300,
            mip_gap=1e-4,
        )
        raw_df["alpha"] = alpha_value
        nd_df["alpha"] = alpha_value
        raw_list.append(raw_df)
        nd_list.append(nd_df)
    alpha_sensitivity_raw_df = pd.concat(raw_list, ignore_index=True)
    alpha_sensitivity_nd_df = pd.concat(nd_list, ignore_index=True)
    save_csv(alpha_sensitivity_raw_df, "sensitivity_alpha_raw_pareto.csv", also_root=True)
    save_csv(alpha_sensitivity_nd_df, "sensitivity_alpha_nondominated_pareto.csv", also_root=True)

alpha_sensitivity_summary = (
    alpha_sensitivity_nd_df
    .groupby("alpha")
    .agg(
        nondominated_points=("total_cost", "count"),
        min_cost=("total_cost", "min"),
        max_cost=("total_cost", "max"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
        min_expected_unmet=("realized_expected_unmet_demand", "min"),
        max_expected_unmet=("realized_expected_unmet_demand", "max"),
    )
    .reset_index()
)
alpha_sensitivity_summary["cost_range"] = alpha_sensitivity_summary["max_cost"] - alpha_sensitivity_summary["min_cost"]
alpha_sensitivity_summary["cvar_range"] = alpha_sensitivity_summary["max_cvar"] - alpha_sensitivity_summary["min_cvar"]
save_csv(alpha_sensitivity_summary, "sensitivity_alpha_summary.csv", also_root=True)

display_if_available(alpha_sensitivity_summary)

In [ ]:
# Cell 19: Plot alpha sensitivity frontier

plt.figure(figsize=(8, 6))
for alpha_value in ALPHA_VALUES:
    temp = alpha_sensitivity_nd_df[alpha_sensitivity_nd_df["alpha"] == alpha_value].copy()
    temp = temp.sort_values("realized_cvar_unmet_demand", ascending=True)
    if temp.empty:
        continue
    plt.plot(
        temp["realized_cvar_unmet_demand"],
        temp["total_cost"],
        marker="o",
        linewidth=1.8,
        label=fr"$\alpha$ = {alpha_value}",
    )
plt.xlabel("Realized CVaR of unmet demand")
plt.ylabel("Total expected cost")
plt.title("Sensitivity Analysis: CVaR Confidence Level")
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sensitivity_alpha_pareto_publication.png", dpi=FIGURE_DPI)
plt.savefig(BASE_DIR / "sensitivity_alpha_pareto_publication.png", dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Cell 20: Sensitivity analysis on hardening effectiveness

hardening_sensitivity_raw_df = maybe_load_csv("sensitivity_hardening_raw_pareto.csv")
hardening_sensitivity_nd_df = maybe_load_csv("sensitivity_hardening_nondominated_pareto.csv")

if hardening_sensitivity_raw_df is None or hardening_sensitivity_nd_df is None:
    raw_list = []
    nd_list = []
    for e_value in HARDENING_EFFECTIVENESS_VALUES:
        method_label = f"ML Dynamic_hardening_{e_value}"
        print("=" * 100)
        print("Running hardening sensitivity:", e_value)
        raw_df, nd_df = generate_pareto_for_risk_method(
            method_name=method_label,
            risk_dict=risk_ml_dynamic,
            alpha=DEFAULT_ALPHA,
            hardening_effectiveness=e_value,
            num_points=NUM_POINTS_SENSITIVITY,
            time_limit=300,
            mip_gap=1e-4,
        )
        raw_df["hardening_effectiveness"] = e_value
        nd_df["hardening_effectiveness"] = e_value
        raw_list.append(raw_df)
        nd_list.append(nd_df)
    hardening_sensitivity_raw_df = pd.concat(raw_list, ignore_index=True)
    hardening_sensitivity_nd_df = pd.concat(nd_list, ignore_index=True)
    save_csv(hardening_sensitivity_raw_df, "sensitivity_hardening_raw_pareto.csv", also_root=True)
    save_csv(hardening_sensitivity_nd_df, "sensitivity_hardening_nondominated_pareto.csv", also_root=True)

hardening_sensitivity_summary = (
    hardening_sensitivity_nd_df
    .groupby("hardening_effectiveness")
    .agg(
        nondominated_points=("total_cost", "count"),
        min_cost=("total_cost", "min"),
        max_cost=("total_cost", "max"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
        min_expected_unmet=("realized_expected_unmet_demand", "min"),
        max_expected_unmet=("realized_expected_unmet_demand", "max"),
    )
    .reset_index()
)
hardening_sensitivity_summary["cost_range"] = hardening_sensitivity_summary["max_cost"] - hardening_sensitivity_summary["min_cost"]
hardening_sensitivity_summary["cvar_range"] = hardening_sensitivity_summary["max_cvar"] - hardening_sensitivity_summary["min_cvar"]
save_csv(hardening_sensitivity_summary, "sensitivity_hardening_summary.csv", also_root=True)

display_if_available(hardening_sensitivity_summary)

In [ ]:
# Cell 21: Plot hardening effectiveness sensitivity frontier

plt.figure(figsize=(8, 6))
for e_value in HARDENING_EFFECTIVENESS_VALUES:
    temp = hardening_sensitivity_nd_df[hardening_sensitivity_nd_df["hardening_effectiveness"] == e_value].copy()
    temp = temp.sort_values("realized_cvar_unmet_demand", ascending=True)
    if temp.empty:
        continue
    plt.plot(
        temp["realized_cvar_unmet_demand"],
        temp["total_cost"],
        marker="o",
        linewidth=1.8,
        label=f"$e$ = {e_value}",
    )
plt.xlabel("Realized CVaR of unmet demand")
plt.ylabel("Total expected cost")
plt.title("Sensitivity Analysis: Hardening Effectiveness")
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sensitivity_hardening_pareto_publication.png", dpi=FIGURE_DPI)
plt.savefig(BASE_DIR / "sensitivity_hardening_pareto_publication.png", dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Cell 22: Extract sensitivity design patterns

alpha_design_patterns = (
    alpha_sensitivity_nd_df
    .groupby(["alpha", "open_dcs", "hardened_dcs"])
    .agg(
        count=("total_cost", "count"),
        min_cost=("total_cost", "min"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
    )
    .reset_index()
    .sort_values(by=["alpha", "count"], ascending=[True, False])
)

hardening_design_patterns = (
    hardening_sensitivity_nd_df
    .groupby(["hardening_effectiveness", "open_dcs", "hardened_dcs"])
    .agg(
        count=("total_cost", "count"),
        min_cost=("total_cost", "min"),
        min_cvar=("realized_cvar_unmet_demand", "min"),
        max_cvar=("realized_cvar_unmet_demand", "max"),
    )
    .reset_index()
    .sort_values(by=["hardening_effectiveness", "count"], ascending=[True, False])
)

save_csv(alpha_design_patterns, "sensitivity_alpha_design_patterns.csv", also_root=True)
save_csv(hardening_design_patterns, "sensitivity_hardening_design_patterns.csv", also_root=True)

print("Alpha sensitivity design patterns:")
display_if_available(alpha_design_patterns, max_rows=30)
print("Hardening sensitivity design patterns:")
display_if_available(hardening_design_patterns, max_rows=30)

In [ ]:
# Cell 23: Validate key results against expected study structure

validation_items = {
    "risk_matrix_rows": len(risk_matrix),
    "reduced_scenarios": risk_matrix["scenario_id"].nunique(),
    "nodes": risk_matrix["node_id"].nunique(),
    "scenario_weight_sum": float(sum(scenario_weights.values())),
    "candidate_designs": len(designs_to_evaluate),
    "common_evaluation_rows": len(common_evaluation_df),
}

if validation_items["reduced_scenarios"] != 29:
    raise RuntimeError(f"Expected 29 reduced scenarios, got {validation_items['reduced_scenarios']}")
if validation_items["risk_matrix_rows"] != 29 * 14:
    raise RuntimeError(f"Expected 406 risk-matrix rows, got {validation_items['risk_matrix_rows']}")
if not np.isclose(validation_items["scenario_weight_sum"], 1.0, atol=1e-6):
    raise RuntimeError(f"Scenario weights do not sum to 1: {validation_items['scenario_weight_sum']}")

print("Validation summary:")
print(json.dumps(validation_items, indent=2))

# A compact checkpoint table for the manuscript-facing numbers.
checkpoint_rows = []
for design_id in ["Fixed_P1", "Historical_P1", "Rule-based_P1", "ML Dynamic_P1", "ML Dynamic_P2", "ML Dynamic_P20"]:
    temp = common_evaluation_df[common_evaluation_df["design_id"] == design_id]
    if not temp.empty:
        row = temp.iloc[0]
        checkpoint_rows.append({
            "design_id": design_id,
            "risk_method": row["risk_method"],
            "evaluated_total_cost": row["evaluated_total_cost"],
            "evaluated_expected_unmet_demand": row["evaluated_expected_unmet_demand"],
            "evaluated_cvar_unmet_demand": row["evaluated_cvar_unmet_demand"],
            "open_dcs": row["open_dcs"],
            "hardened_dcs": row["hardened_dcs"],
        })
checkpoint_df = pd.DataFrame(checkpoint_rows)
save_csv(checkpoint_df, "optimization_common_evaluation_checkpoint.csv", also_root=True)
display_if_available(checkpoint_df)

In [ ]:
# Cell 24: Final output manifest for Notebook 02

key_output_files = [
    "optimization_node_parameters.csv",
    "optimization_transport_arcs.csv",
    "risk_method_summary.csv",
    "gurobi_optimization_anchor_solutions.csv",
    "gurobi_pareto_frontier_ml_dynamic_risk_realized_cvar.csv",
    "gurobi_pareto_frontier_ml_dynamic_risk_nondominated.csv",
    "corrected_pareto_frontier_ml_dynamic_risk.png",
    "corrected_pareto_design_transition_summary.csv",
    "raw_pareto_comparison_all_risk_methods.csv",
    "nondominated_pareto_comparison_all_risk_methods.csv",
    "pareto_comparison_diagnostic.csv",
    "representative_designs_by_risk_method.csv",
    "designs_to_common_evaluate.csv",
    "common_evaluation_under_ml_dynamic_risk.csv",
    "common_evaluation_nondominated_designs_under_ml_dynamic.csv",
    "common_evaluation_summary_by_risk_method.csv",
    "common_evaluation_improvement_summary.csv",
    "table_main_common_evaluation_improvement.csv",
    "table_representative_designs_common_evaluation.csv",
    "publication_common_evaluation_designs_clean_v2.png",
    "bar_cvar_reduction_over_baselines.png",
    "evaluation_environment_summary.csv",
    "multi_environment_common_evaluation.csv",
    "multi_environment_common_evaluation_summary.csv",
    "best_designs_multi_environment.csv",
    "sensitivity_alpha_raw_pareto.csv",
    "sensitivity_alpha_nondominated_pareto.csv",
    "sensitivity_alpha_summary.csv",
    "sensitivity_alpha_pareto_publication.png",
    "sensitivity_hardening_raw_pareto.csv",
    "sensitivity_hardening_nondominated_pareto.csv",
    "sensitivity_hardening_summary.csv",
    "sensitivity_hardening_pareto_publication.png",
    "sensitivity_alpha_design_patterns.csv",
    "sensitivity_hardening_design_patterns.csv",
    "optimization_common_evaluation_checkpoint.csv",
]

manifest = pd.DataFrame({
    "file_name": key_output_files,
    "exists_in_project_root": [(BASE_DIR / f).exists() for f in key_output_files],
    "exists_in_output_dir": [(OUTPUT_DIR / f).exists() for f in key_output_files],
})
save_csv(manifest, "key_output_files_for_manuscript_02.csv", also_root=True)

display_if_available(manifest, max_rows=100)
print("Notebook 02 pipeline completed.")

In [ ]:

# PHASE 2 Gurobi extension cell.
# Run this after Notebook 02 has defined solve_resilient_supply_chain_gurobi,
# generate_pareto_for_risk_method, evaluate_fixed_design_gurobi, risk_methods,
# scenarios, all_nodes, and design utilities.
# It adds Coastal/Inland and Engineered Logistic risk representations to the optimization comparison.

conference_risk_path = BASE_DIR / "conference_optimization_risk_matrix_reduced_with_baselines.csv"
conference_risk = pd.read_csv(conference_risk_path)

risk_coastal_inland = conference_risk.set_index(["scenario_id", "node_id"])["p_coastal_inland"].astype(float).to_dict()
risk_engineered_logistic = conference_risk.set_index(["scenario_id", "node_id"])["p_engineered_logistic_calibrated"].astype(float).to_dict()

# Add to design-generation risk methods.
risk_methods_conference = dict(risk_methods)
risk_methods_conference["Coastal/Inland"] = risk_coastal_inland
risk_methods_conference["Engineered Logistic"] = risk_engineered_logistic

raw_list = []
nd_list = []
for method_name, risk_dict in risk_methods_conference.items():
    raw_df, nd_df = generate_pareto_for_risk_method(
        method_name=method_name,
        risk_dict=risk_dict,
        alpha=DEFAULT_ALPHA,
        num_points=12,        # conference version: compact epsilon grid
        time_limit=300,
        mip_gap=1e-4,
    )
    raw_list.append(raw_df)
    nd_list.append(nd_df)

conference_raw_all = pd.concat(raw_list, ignore_index=True)
conference_nd_all = pd.concat(nd_list, ignore_index=True)
save_csv(conference_raw_all, "conference_raw_pareto_all_risk_representations.csv", also_root=True)
save_csv(conference_nd_all, "conference_nondominated_pareto_all_risk_representations.csv", also_root=True)

# Build candidate designs from nondominated solutions.
conference_design_rows = []
for _, row in conference_nd_all.iterrows():
    conference_design_rows.append({
        "design_id": f"{row['risk_method']}_P{int(row['point_id'])}",
        "risk_method": row["risk_method"],
        "source_point_id": int(row["point_id"]),
        "open_dcs": row["open_dcs"],
        "hardened_dcs": row["hardened_dcs"],
        "internal_total_cost": row["total_cost"],
        "internal_realized_cvar": row["realized_cvar_unmet_demand"],
    })
conference_designs_to_evaluate = pd.DataFrame(conference_design_rows).drop_duplicates(
    subset=["risk_method", "open_dcs", "hardened_dcs"]
)
save_csv(conference_designs_to_evaluate, "conference_designs_to_evaluate.csv", also_root=True)

# Full cross-environment table on reduced scenario set.
evaluation_environments_conference = {
    "Fixed": risk_fixed,
    "Historical": risk_historical,
    "Rule_binary": p_rule_binary,
    "Engineered_Logistic": risk_engineered_logistic,
    "RF_calibrated": risk_ml_dynamic,
    "RF_uncalibrated": p_is_sensitivity,
}

multi_rows = []
for env_name, env_risk in evaluation_environments_conference.items():
    for _, row in conference_designs_to_evaluate.iterrows():
        open_set = parse_design_string(row["open_dcs"])
        harden_set = parse_design_string(row["hardened_dcs"])
        result = evaluate_fixed_design_gurobi(
            open_dcs_set=open_set,
            hardened_dcs_set=harden_set,
            evaluation_risk_dict=env_risk,
            alpha=DEFAULT_ALPHA,
            time_limit=300,
            mip_gap=1e-4,
            output_flag=0,
        )
        multi_rows.append({
            "evaluation_environment": env_name,
            "design_id": row["design_id"],
            "risk_method": row["risk_method"],
            "open_dcs": design_to_string(open_set),
            "hardened_dcs": design_to_string(harden_set),
            **result,
        })
conference_multi_environment = pd.DataFrame(multi_rows)
save_csv(conference_multi_environment, "conference_multi_environment_evaluation.csv", also_root=True)

conference_multi_summary = (
    conference_multi_environment
    .groupby(["evaluation_environment", "risk_method"])
    .agg(
        design_count=("design_id", "count"),
        min_cost=("evaluated_total_cost", "min"),
        min_cvar=("evaluated_cvar_unmet_demand", "min"),
        min_expected_unmet=("evaluated_expected_unmet_demand", "min"),
    )
    .reset_index()
)
save_csv(conference_multi_summary, "conference_multi_environment_summary.csv", also_root=True)
display(conference_multi_summary)


In [ ]:
# ======================================================================================
# Conference add-on: Full 464-scenario re-evaluation of representative designs
# ======================================================================================
# Purpose:
#   Fix first-stage designs generated from reduced-scenario optimization,
#   then re-optimize second-stage recourse under all 464 historical hurricane scenarios.
#
# This creates:
#   conference_full464_evaluation.csv
#   conference_full464_summary.csv
#
# Run this in Notebook 02 after:
#   - global setup
#   - node/arc/model parameter setup
#   - utility functions
#   - solve_resilient_supply_chain_gurobi(...)
# ======================================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# If BASE_DIR was already defined in Notebook 02, this keeps it.
# Otherwise, manually set it.
try:
    BASE_DIR
except NameError:
    BASE_DIR = PROJECT_ROOT

BASE_DIR = Path(BASE_DIR)
OUTPUT_DIR = BASE_DIR / "outputs_04_conference_revision"
OUTPUT_DIR.mkdir(exist_ok=True)

full_file = BASE_DIR / "conference_scenario_node_probabilities_full_with_baselines.csv"
design_file = BASE_DIR / "conference_designs_to_evaluate.csv"

if not full_file.exists():
    raise FileNotFoundError(f"Missing full scenario probability file: {full_file}")
if not design_file.exists():
    raise FileNotFoundError(f"Missing design file: {design_file}")

full_df = pd.read_csv(full_file)
designs_df = pd.read_csv(design_file)

print("Full scenario-node probability file:", full_df.shape)
print("Unique full scenarios:", full_df["scenario_id"].nunique())
print("Unique nodes:", full_df["node_id"].nunique())
print("Designs to evaluate:", designs_df.shape)

# -----------------------------
# Build full 464 scenario set
# -----------------------------
full_scenarios = full_df["scenario_id"].drop_duplicates().tolist()

full_scenario_weights = (
    full_df[["scenario_id", "scenario_weight"]]
    .drop_duplicates()
    .set_index("scenario_id")["scenario_weight"]
    .astype(float)
    .to_dict()
)

# Normalize just in case of tiny floating error
w_sum = sum(full_scenario_weights.values())
full_scenario_weights = {s: w / w_sum for s, w in full_scenario_weights.items()}

print("Full scenario weight sum:", sum(full_scenario_weights.values()))

# -----------------------------
# Build full-environment risk dictionaries
# -----------------------------
def build_full_risk_dict(col):
    return full_df.set_index(["scenario_id", "node_id"])[col].astype(float).to_dict()

full_risk_envs = {
    "Full464_RF_calibrated": build_full_risk_dict("p_is_calibrated"),
    "Full464_RF_uncalibrated": build_full_risk_dict("p_is_uncalibrated"),
    "Full464_Engineered_Logistic": build_full_risk_dict("p_engineered_logistic_calibrated"),
    "Full464_Coastal_Inland": build_full_risk_dict("p_coastal_inland"),
    "Full464_Rule_binary": build_full_risk_dict("disrupted"),
}

# Optional: full fixed and historical environments over 464 scenarios.
# Fixed = same smoothed global exposure rate for every scenario-node.
global_positive = float(full_df["disrupted"].sum())
global_total = float(full_df.shape[0])
fixed_probability_full = (global_positive + 1.0) / (global_total + 2.0)

full_risk_envs["Full464_Fixed"] = {
    (s, i): fixed_probability_full
    for s in full_scenarios
    for i in full_df["node_id"].drop_duplicates().tolist()
}

node_freq = (
    full_df.groupby("node_id")["disrupted"]
    .agg(["sum", "count"])
    .reset_index()
)
node_freq["historical_probability"] = (node_freq["sum"] + 1.0) / (node_freq["count"] + 2.0)
node_hist_prob = node_freq.set_index("node_id")["historical_probability"].to_dict()

full_risk_envs["Full464_Historical"] = {
    (s, i): float(node_hist_prob.get(i, fixed_probability_full))
    for s in full_scenarios
    for i in full_df["node_id"].drop_duplicates().tolist()
}

# -----------------------------
# IMPORTANT:
# solve_resilient_supply_chain_gurobi uses global variables:
#   scenarios
#   scenario_weights
# So we temporarily replace them with the full 464-scenario set.
# -----------------------------
old_scenarios = scenarios
old_scenario_weights = scenario_weights

full_eval_rows = []

try:
    scenarios = full_scenarios
    scenario_weights = full_scenario_weights

    print("Temporarily switched optimization model to full 464 scenarios.")
    print("Number of scenarios used by solver:", len(scenarios))

    # To save time for the conference paper, start with the most important environments.
    # You can reduce this list if runtime is too long.
    env_order = [
        "Full464_RF_calibrated",
        "Full464_Engineered_Logistic",
        "Full464_Rule_binary",
        "Full464_Fixed",
        "Full464_Historical",
        "Full464_RF_uncalibrated",
        "Full464_Coastal_Inland",
    ]

    for env_name in env_order:
        env_risk = full_risk_envs[env_name]
        print("=" * 100)
        print("Full-464 evaluation environment:", env_name)

        for _, row in designs_df.iterrows():
            design_id = row["design_id"]
            risk_method = row["risk_method"]
            open_dcs = row["open_dcs"]
            hardened_dcs = row["hardened_dcs"]

            print(f"Evaluating {design_id} under {env_name}...")

            sol = solve_resilient_supply_chain_gurobi(
                objective="cost",
                risk_dict=env_risk,
                fixed_open_dcs=open_dcs,
                fixed_hardened_dcs=hardened_dcs,
                alpha=DEFAULT_ALPHA,
                hardening_effectiveness=MITIGATION_EFFECTIVENESS,
                unmet_penalty=UNMET_PENALTY_PER_UNIT,
                time_limit=300,
                mip_gap=1e-4,
                output_flag=0,
            )

            full_eval_rows.append({
                "evaluation_environment": env_name,
                "design_id": design_id,
                "risk_method": risk_method,
                "open_dcs": open_dcs,
                "hardened_dcs": hardened_dcs,
                "status": sol.get("status"),
                "evaluated_total_cost": sol.get("total_cost"),
                "evaluated_fixed_cost": sol.get("fixed_cost"),
                "evaluated_hardening_cost": sol.get("hardening_cost"),
                "evaluated_transport_cost": sol.get("expected_transport_cost"),
                "evaluated_unmet_penalty": sol.get("expected_unmet_penalty"),
                "evaluated_disruption_penalty": sol.get("expected_disruption_penalty"),
                "evaluated_expected_unmet_demand": sol.get("realized_expected_unmet_demand"),
                "evaluated_cvar_unmet_demand": sol.get("realized_cvar_unmet_demand"),
                "evaluated_eta_cvar": sol.get("realized_eta_cvar"),
            })

finally:
    # Restore reduced-scenario globals so the rest of Notebook 02 is not affected.
    scenarios = old_scenarios
    scenario_weights = old_scenario_weights
    print("Restored original reduced-scenario settings.")

full464_eval = pd.DataFrame(full_eval_rows)

full464_eval.to_csv(BASE_DIR / "conference_full464_evaluation.csv", index=False)
full464_eval.to_csv(OUTPUT_DIR / "conference_full464_evaluation.csv", index=False)

full464_summary = (
    full464_eval
    .groupby(["evaluation_environment", "risk_method"], as_index=False)
    .agg(
        design_count=("design_id", "count"),
        min_cost=("evaluated_total_cost", "min"),
        min_cvar=("evaluated_cvar_unmet_demand", "min"),
        min_expected_unmet=("evaluated_expected_unmet_demand", "min"),
    )
)

full464_summary.to_csv(BASE_DIR / "conference_full464_summary.csv", index=False)
full464_summary.to_csv(OUTPUT_DIR / "conference_full464_summary.csv", index=False)

print("Saved:")
print(BASE_DIR / "conference_full464_evaluation.csv")
print(BASE_DIR / "conference_full464_summary.csv")

display(full464_summary)